# Step-by-step tutorial: vanilla library vs weak library vs proposed weak-Pareto method

This notebook is intentionally written like a PDE-FIND tutorial: we first inspect the data and candidate matrices by hand, then we call the same method wrappers used by the benchmark scripts.

**Final paper framing**

- **Proposed method:** `weak_pareto` = weak candidate library + best-subset Pareto-DE.
- **Baseline 1:** `vanilla_pareto` = vanilla/pointwise candidate library + best-subset Pareto-DE.
- **Baseline 2:** `weak_grid_stridge` = weak candidate library + grid-search STRidge.
- **Baseline 3:** `weak_fixed_stability` = fixed weak grid library + stability-selected STRidge. This is a fixed-library ablation, not the proposed continuous-order Pareto-DE method.

The canonical paper candidate class is deliberately overcomplete: terms are `u^p D_x^beta u` with `p_values=(0,1,2)` and `cmax=4`. This is broader than the true bundled linear two-term equations; use `cmax_override=5` only for a heavier stress test.

The important consistency rule is: **the notebook and the scripts use the same `benchmark_spec(...)` configuration and the same method dispatcher.** If you set the same dataset, noise, profile, seed, and runtime options here, you should obtain the same result as `scripts/run_all_methods.py`.


## 0. User controls

Change this first cell when you want to reproduce a specific paper case.

For final paper settings, use:

```python
profile = "paper"
maxiter_override = None
popsize_override = None
```

When `profile="paper"`, the dataset-specific paper budgets are chosen automatically. Only set `maxiter_override` or `popsize_override` for debugging or ablation.


In [ ]:
# Robust project-root discovery.
# This avoids failures when a Jupyter kernel is started in a directory that is
# later moved/deleted, in which case Path.cwd() itself can raise FileNotFoundError.
import os
import sys
from pathlib import Path


def find_fpde_project_root() -> Path:
    """Return the repository root containing dataset_configs.py and data/.

    Priority:
    1. FPDE_PROJECT_ROOT environment variable, if set.
    2. Current/PWD directories and their parents, if available.
    3. Common local search locations. This keeps notebooks runnable from
       project root, from notebooks/, and after opening a notebook from an IDE.
    """
    def looks_like_root(path: Path) -> bool:
        return (
            (path / "dataset_configs.py").is_file()
            and (path / "weak_pareto_fde_discovery.py").is_file()
            and (path / "data").is_dir()
        )

    candidates = []
    env_root = os.environ.get("FPDE_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    # os.getcwd() can fail if the kernel's working directory was deleted.
    try:
        candidates.append(Path(os.getcwd()).expanduser())
    except FileNotFoundError:
        pass

    # PWD may still contain a useful absolute path even when os.getcwd() fails.
    pwd = os.environ.get("PWD")
    if pwd:
        candidates.append(Path(pwd).expanduser())

    # Also try the directory containing this notebook if Jupyter exposes it via env.
    for key in ("NOTEBOOK_DIR", "JUPYTER_SERVER_ROOT"):
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    seen = set()
    for cand in candidates:
        try:
            cand = cand.resolve(strict=False)
        except Exception:
            continue
        for path in (cand, cand.parent, *cand.parents):
            if path in seen:
                continue
            seen.add(path)
            if looks_like_root(path):
                return path

    # Conservative bounded search over common project locations.
    search_roots = [
        Path.home() / "Desktop" / "research",
        Path.home() / "Desktop",
        Path.home(),
        Path("/mnt/data"),
    ]
    max_dirs = 5000
    for base in search_roots:
        if not base.exists():
            continue
        visited = 0
        for dirpath, dirnames, filenames in os.walk(base):
            visited += 1
            # Keep the search cheap and avoid hidden/cache directories.
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in {"__pycache__", ".ipynb_checkpoints"}]
            if "dataset_configs.py" in filenames and "weak_pareto_fde_discovery.py" in filenames:
                candidate = Path(dirpath)
                if looks_like_root(candidate):
                    return candidate
            if visited >= max_dirs:
                break

    raise FileNotFoundError(
        "Could not locate the fractional_pareto project root. "
        "Set FPDE_PROJECT_ROOT=/path/to/fractional_pareto_publication_ready_final "
        "or open the notebook from the project root/notebooks directory."
    )


ROOT = find_fpde_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")


import json
from argparse import Namespace

import numpy as np
import pandas as pd

from dataset_configs import benchmark_spec, available_benchmark_dataset_names, config_search_space_fingerprint
from pareto_fde_discovery import FractionalFeatureBank
from weak_pareto_fde_discovery import WeakFractionalFeatureBank, run_weak_pareto_discovery, model_order_metrics
from scripts.run_all_methods import run_single_method, run_all_methods, selected_model_dict

# EDIT THESE VALUES
dataset_name = "synthetic_time_space_fractional_RD"
noise_percent = 0.5
profile = "notebook"  # change to "paper" for final-quality settings
seed = 0

# Candidate-library overrides. In notebook mode we use a small teaching library;
# in paper mode, leave them as None to use the canonical paper library.
cmax_override = 2 if profile == "notebook" else None
p_values_override = (0,) if profile == "notebook" else None

# Runtime overrides. In paper mode, leave these as None to use dataset-specific paper defaults.
maxiter_override = 0 if profile == "notebook" else None
popsize_override = 2 if profile == "notebook" else None

# Method comparison controls. These are the same labels accepted by scripts/run_all_methods.py.
methods = ["weak_pareto", "vanilla_pareto", "weak_grid_stridge", "weak_fixed_stability"]
weak_test_budget = "smoke" if profile == "notebook" else "paper"
stability_splits = 1 if profile == "notebook" else 5
stability_width_scales = [1.0] if profile == "notebook" else [0.8, 1.0, 1.2]
out_root = ROOT / "results" / "notebook_01_step_by_step"
out_root.mkdir(parents=True, exist_ok=True)

print("Available datasets:")
for name in available_benchmark_dataset_names():
    print("  -", name)


## 1. Load the benchmark spec

`benchmark_spec(...)` is the single source of truth for a dataset case. It returns:

- `data`: the clean benchmark dataset;
- `config`: the shared discovery configuration used by all methods;
- `truth_spec`: post-hoc truth metadata for scoring only;
- `config_search_space`: the candidate search space fingerprint.

The data object is intentionally clean. The requested noise level is stored in `config.noise_percent`, so each method injects the same noise realization internally using `config.seed`. This prevents accidental double-noising.


In [ ]:
spec = benchmark_spec(
    dataset_name,
    data_dir=ROOT / "data",
    profile=profile,
    noise_percent=noise_percent,
    seed=seed,
    maxiter=maxiter_override,
    popsize=popsize_override,
    cmax=cmax_override,
    p_values=p_values_override,
)

data = spec["data"]
config = spec["config"]
truth_spec = spec["truth_spec"]
truth_terms = [(int(p), float(b)) for p, b in spec["expected_terms"]]
truth_alpha = float(spec["expected_alpha"])

config.progress = False
config.progress_de = False

print("Dataset:", data.name)
print("Truth:", data.truth)
print("U shape:", data.U.shape)
print("dt, dx:", data.dt, data.dx)
print("Expected alpha:", truth_alpha)
print("Expected RHS terms:", truth_terms)
print("Runtime DE budget: maxiter=", config.maxiter, "popsize=", config.popsize)
print("Candidate space fingerprint:")
print(json.dumps(config_search_space_fingerprint(config), indent=2)[:3000])

## 2. Inspect the clean and noisy observations

The discovery methods receive a noisy realization internally. The exact same noise realization is used for vanilla, weak, STRidge, and stability baselines because they share `config.noise_percent` and `config.seed`.


In [ ]:
rng = np.random.default_rng(config.seed)
U_clean = data.U
sigma = (config.noise_percent / 100.0) * np.std(U_clean)
U_noisy_preview = U_clean + sigma * rng.standard_normal(U_clean.shape)

print("noise_percent:", config.noise_percent)
print("noise sigma:", sigma)
print("clean U: min/max/std", float(U_clean.min()), float(U_clean.max()), float(U_clean.std()))
print("noisy preview: min/max/std", float(U_noisy_preview.min()), float(U_noisy_preview.max()), float(U_noisy_preview.std()))

# A compact table is more robust than plotting in all environments.
pd.DataFrame({
    "clean_first_time": U_clean[0, :8],
    "noisy_first_time": U_noisy_preview[0, :8],
})

## 3. Vanilla candidate library construction

The vanilla/strong library approximates pointwise fractional derivatives of noisy data.

For a chosen alpha and true-like beta list, the matrix has the form

\[
 y_i pprox D_t^lpha u_i, \qquad
 \Theta_{ij} pprox u_i^{p_j} D_x^{eta_j}u_i.
\]

This is the baseline candidate construction used by `vanilla_pareto`. It is fragile under noise because the derivative operator is applied directly to the observed field.


In [ ]:
vanilla_bank = FractionalFeatureBank(data, config)
vanilla_bank.precompute(verbose=False)

probe_betas = [beta for _, beta in truth_terms]
y_vanilla = vanilla_bank.target(truth_alpha)
Theta_vanilla = np.column_stack([
    vanilla_bank.u_power(0) * vanilla_bank.spatial(beta)
    for beta in probe_betas
])

print("vanilla target shape:", y_vanilla.shape)
print("vanilla RHS matrix shape:", Theta_vanilla.shape)
print("first 5 target entries:", np.round(y_vanilla[:5], 6))
print("first 5 RHS rows:")
pd.DataFrame(Theta_vanilla[:5], columns=[f"D_x^{b:g} u" for b in probe_betas])

## 4. Weak candidate library construction

The weak library uses the same symbolic terms but changes how the columns are computed.

Instead of differentiating noisy data, it uses fractional adjoints on smooth test functions:

\[
\langle D_x^eta u,\phiangle = \langle u,(D_x^eta)^*\phiangle.
\]

So each row is an integral feature. This is the core noise-robustness mechanism of the proposed method.


In [ ]:
weak_bank = WeakFractionalFeatureBank(data, config, test_budget=weak_test_budget)
weak_bank.precompute(verbose=False)

y_weak = weak_bank.target(truth_alpha)
Theta_weak = np.column_stack([
    weak_bank.spatial_feature(0, beta)
    for beta in probe_betas
])

print("weak target shape:", y_weak.shape)
print("weak RHS matrix shape:", Theta_weak.shape)
print("number of weak rows:", weak_bank.n_points)
print("time test functions:", weak_bank.time_tests.shape)
print("space test functions:", weak_bank.space_tests.shape)
print("first 5 weak target entries:", np.round(y_weak[:5], 6))
print("first 5 weak RHS rows:")
pd.DataFrame(Theta_weak[:5], columns=[f"weak D_x^{b:g} u" for b in probe_betas])

## 5. Simple diagnostic: column correlation

Fractional derivative columns can be highly correlated, especially for nearby beta values. Noise makes this worse. The weak library usually reduces derivative noise, but it cannot remove all identifiability issues.


In [ ]:
def column_corr(X):
    X = np.asarray(X, dtype=float)
    X = X - X.mean(axis=0, keepdims=True)
    X = X / (np.linalg.norm(X, axis=0, keepdims=True) + 1e-14)
    return X.T @ X

print("Vanilla candidate correlation:")
display(pd.DataFrame(np.round(column_corr(Theta_vanilla), 4), columns=probe_betas, index=probe_betas))

print("Weak candidate correlation:")
display(pd.DataFrame(np.round(column_corr(Theta_weak), 4), columns=probe_betas, index=probe_betas))

## 6. Run the proposed method and baselines using the same dispatcher as the script

This is the main consistency check. The loop below calls `run_single_method(...)`, which is the same method dispatcher used by `scripts/run_all_methods.py`.

Therefore, with the same controls from Section 0, these results should match the command-line benchmark for this one dataset/noise/seed case.


In [ ]:
method_results = {}
selected_models = {}

for method in methods:
    print(f"\nRunning {method}...")
    result = run_single_method(
        method,
        data,
        config,
        out_root / method,
        weak_test_budget=weak_test_budget,
        stability_splits=stability_splits,
        stability_width_scales=stability_width_scales,
        verbose=False,
    )
    selected = selected_model_dict(method, result)
    method_results[method] = result
    selected_models[method] = selected
    print(selected["equation"])


## 7. Score structure, fractional-order error, and coefficient diagnostics

The scoring below is post-hoc. The methods did not use the truth metadata during fitting.

Key fields:

- `full_structure_recovered`: symbolic support/form recovery only. It does **not** require small alpha, beta, or coefficient errors.
- `rhs_f1`: F1 score for symbolic RHS term recovery.
- `alpha_abs_error`: absolute error of the temporal fractional order.
- `max_matched_beta_abs_error`: largest matched RHS fractional-order error.
- `selected_coefficients`: coefficients of the selected equation; coefficient-error tables are produced by the benchmark script.
- `val_rel_mse`: validation residual normalized by target variance.

In [ ]:
rows = []
for method, selected in selected_models.items():
    metrics = model_order_metrics(
        selected,
        truth_alpha,
        truth_terms,
        alpha_tol=truth_spec.alpha_tol,
        beta_tol=truth_spec.beta_tol,
    )
    rows.append({
        "method": method,
        "role": "proposed" if method == "weak_pareto" else "baseline",
        "equation": selected["equation"],
        "full_structure_recovered": metrics["full_structure_recovered"],
        "rhs_f1": metrics["rhs_f1"],
        "alpha_abs_error": metrics["alpha_abs_error"],
        "max_matched_beta_abs_error": metrics["max_matched_beta_abs_error"],
        "selected_alpha": selected.get("alpha"),
        "selected_terms": selected.get("terms"),
        "selected_coefficients": selected.get("coefficients"),
        "val_rel_mse": selected.get("val_rel_mse"),
    })

summary_df = pd.DataFrame(rows)
summary_df

## 8. Equivalent command-line run for this exact case

The following cell calls `run_all_methods(...)` with one dataset, one noise level, and one seed. It is the Python equivalent of running `scripts/run_all_methods.py` from the terminal.

Use this when you want to confirm notebook/script consistency. It writes the same `method_comparison.csv` format used by the paper benchmark scripts.


In [ ]:
script_equiv_out = out_root / "script_equivalent"
args = Namespace(
    data_dir=ROOT / "data",
    output_dir=script_equiv_out,
    datasets=[dataset_name],
    methods=methods,
    noise_levels=[noise_percent],
    seeds=[seed],
    profile=profile,
    maxiter=maxiter_override,
    popsize=popsize_override,
    cmax=cmax_override,
    p_values=p_values_override,
    weak_test_budget=weak_test_budget,
    stability_splits=stability_splits,
    stability_width_scales=list(stability_width_scales),
    quiet=True,
    progress=False,
    progress_de=False,
)

script_rows = run_all_methods(args)
script_df = pd.DataFrame(script_rows)
script_df[[
    "dataset", "noise_percent", "method", "selected_equation",
    "full_structure_recovered", "rhs_f1", "alpha_abs_error",
    "max_matched_beta_abs_error", "max_coef_rel_error", "val_rel_mse"
]]

## 9. Terminal command with the same settings

The equivalent terminal command is printed below. For `profile='paper'`, omit `--maxiter` and `--popsize` unless you intentionally set overrides.


In [ ]:
cmd = [
    "python", "scripts/run_all_methods.py",
    "--datasets", dataset_name,
    "--methods", *methods,
    "--profile", profile,
    "--noise-levels", str(noise_percent),
    "--seeds", str(seed),
    "--weak-test-budget", weak_test_budget,
    "--stability-splits", str(stability_splits),
    "--stability-width-scales", *[str(v) for v in stability_width_scales],
    "--quiet",
    "--output-dir", str(script_equiv_out),
]
if maxiter_override is not None:
    cmd += ["--maxiter", str(maxiter_override)]
if popsize_override is not None:
    cmd += ["--popsize", str(popsize_override)]
print(" \
  ".join(cmd))

## 10. What to remember

1. Set `dataset_name`, `noise_percent`, `profile`, and `seed` in Section 0.
2. The notebook calls `benchmark_spec(...)`, the same API used by the scripts.
3. The method loop calls `run_single_method(...)`, the same dispatcher used by `run_all_methods.py`.
4. `weak_pareto` is the proposed method: **weak library + best-subset Pareto-DE**.
5. The baselines isolate different effects:
   - `vanilla_pareto`: same optimizer, vanilla library;
   - `weak_grid_stridge`: same weak library, STRidge selector;
   - `weak_fixed_stability`: fixed-grid weak stability ablation.

So if the controls are the same, notebook and script results should be consistent.
